In [21]:
!python --version

Python 3.12.12


In [1]:
# Only install what Colab does NOT already have.
# DO NOT upgrade numpy, scipy, pandas, or scikit-learn!
# Colab's pre-installed versions are already compatible with each other.
!pip install kagglehub -q

print("\n" + "="*50)
print("✅  kagglehub installed. All other deps are pre-installed.")
print("="*50)


✅  kagglehub installed. All other deps are pre-installed.


In [14]:
import kagglehub
import shutil
import os

DATASET_SLUG = "bcccdatasets/intrusion-detection-datasets-bccc-cic-ids-2017"
DATA_DIR = "data"

# Download (kagglehub caches it; returns the cache path)
print("[*] Downloading CICIDS 2017 dataset via kagglehub...")
dataset_path = kagglehub.dataset_download(DATASET_SLUG)
print(f"    Downloaded to: {dataset_path}")

# Copy CSV files into a local data/ folder
os.makedirs(DATA_DIR, exist_ok=True)
csv_count = 0
for root, _dirs, files in os.walk(dataset_path):
    for f in files:
        if f.endswith(".csv"):
            src = os.path.join(root, f)
            dst = os.path.join(DATA_DIR, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            csv_count += 1

print(f"\n{csv_count} CSV file(s) ready in '{DATA_DIR}/'")

# List the files
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.csv'):
        size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024*1024)
        print(f"    {f} ({size_mb:.1f} MB)")

[*] Downloading CICIDS 2017 dataset via kagglehub...
    Downloaded to: /kaggle/input/intrusion-detection-datasets-bccc-cic-ids-2017

18 CSV file(s) ready in 'data/'
    botnet_ares.csv (3.8 MB)
    ddos_loit.csv (86.9 MB)
    dos_golden_eye.csv (7.7 MB)
    dos_hulk.csv (261.8 MB)
    dos_slowhttptest.csv (5.0 MB)
    dos_slowloris.csv (4.1 MB)
    friday_benign.csv (278.6 MB)
    ftp_patator.csv (7.1 MB)
    heartbleed.csv (0.0 MB)
    monday_benign.csv (377.2 MB)
    portscan.csv (112.8 MB)
    ssh_patator-new.csv (4.7 MB)
    thursday_benign.csv (102.9 MB)
    tuesday_benign.csv (303.2 MB)
    web_brute_force.csv (1.8 MB)
    web_sql_injection.csv (0.0 MB)
    web_xss.csv (0.9 MB)
    wednesday_benign.csv (303.6 MB)


In [3]:
!pip install scikit-learn

In [15]:
import json
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

# Top 20 most discriminative features from CICIDS 2017
# Column names match the Kaggle dataset (snake_case format)
TOP_20_FEATURES = [
    "duration",
    "fwd_packets_count",
    "bwd_packets_count",
    "fwd_total_payload_bytes",
    "bwd_total_payload_bytes",
    "fwd_payload_bytes_max",
    "fwd_payload_bytes_mean",
    "bwd_payload_bytes_max",
    "bwd_payload_bytes_mean",
    "bytes_rate",
    "packets_rate",
    "packets_IAT_mean",
    "packet_IAT_max",
    "fwd_packets_IAT_total",
    "fwd_packets_IAT_mean",
    "bwd_packets_IAT_total",
    "bwd_packets_IAT_mean",
    "fwd_psh_flag_counts",
    "fwd_packets_rate",
    "bwd_packets_rate",
]

# Simplified attack category mapping
ATTACK_MAPPING = {
    "BENIGN": "BENIGN",
    "FTP-Patator": "Brute Force",
    "SSH-Patator": "Brute Force",
    "DoS slowloris": "DDoS",
    "DoS Slowhttptest": "DDoS",
    "DoS Hulk": "DDoS",
    "DoS GoldenEye": "DDoS",
    "Heartbleed": "DDoS",
    "Web Attack \u0096 Brute Force": "Web Attack",
    "Web Attack \u0096 XSS": "Web Attack",
    "Web Attack \u0096 Sql Injection": "SQL Injection",
    "Web Attack \u2013 Brute Force": "Web Attack",
    "Web Attack \u2013 XSS": "Web Attack",
    "Web Attack \u2013 Sql Injection": "SQL Injection",
    "Infiltration": "Infiltration",
    "Bot": "Bot",
    "PortScan": "PortScan",
    "DDoS": "DDoS",
}

# Severity levels
SEVERITY_MAPPING = {
    "BENIGN": "Low",
    "PortScan": "Medium",
    "Brute Force": "High",
    "DDoS": "High",
    "Web Attack": "High",
    "SQL Injection": "High",
    "Bot": "Medium",
    "Infiltration": "High",
}

MODELS_DIR = "models"
DATA_DIR = "data"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Constants & imports ready.")

Constants & imports ready.


In [16]:
def load_and_clean_data(data_dir):
    """Load all CSVs from data_dir, clean labels, map attacks."""
    print("[*] Loading dataset files...")
    csv_files = sorted([f for f in os.listdir(data_dir) if f.endswith(".csv")])
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    frames = []
    for csv_file in csv_files:
        filepath = os.path.join(data_dir, csv_file)
        print(f"    Loading: {csv_file}")
        try:
            df = pd.read_csv(filepath, encoding="utf-8", low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(filepath, encoding="latin-1", low_memory=False)
        frames.append(df)

    data = pd.concat(frames, ignore_index=True)
    print(f"\n[*] Total records loaded: {len(data):,}")

    # Normalise column names
    data.columns = data.columns.str.strip()

    # Find label column
    label_col = next(
        (c for c in [" Label", "Label", "label"] if c in data.columns), None
    )
    if label_col is None:
        raise ValueError("Label column not found in dataset")
    data = data.rename(columns={label_col: "Label"})
    data["Label"] = data["Label"].str.strip()

    print("\n[*] Label distribution (raw):")
    print(data["Label"].value_counts())

    # Map to simplified categories
    data["Label"] = data["Label"].map(ATTACK_MAPPING).fillna("Other")
    data = data[data["Label"] != "Other"]

    print("\n[*] Label distribution (mapped):")
    print(data["Label"].value_counts())

    return data


data = load_and_clean_data(DATA_DIR)

[*] Loading dataset files...
    Loading: botnet_ares.csv
    Loading: ddos_loit.csv
    Loading: dos_golden_eye.csv
    Loading: dos_hulk.csv
    Loading: dos_slowhttptest.csv
    Loading: dos_slowloris.csv
    Loading: friday_benign.csv
    Loading: ftp_patator.csv
    Loading: heartbleed.csv
    Loading: monday_benign.csv
    Loading: portscan.csv
    Loading: ssh_patator-new.csv
    Loading: thursday_benign.csv
    Loading: tuesday_benign.csv
    Loading: web_brute_force.csv
    Loading: web_sql_injection.csv
    Loading: web_xss.csv
    Loading: wednesday_benign.csv

[*] Total records loaded: 2,438,052

[*] Label distribution (raw):
Label
Benign               1786239
DoS_Hulk              349240
Port_Scan             161323
DDoS_LOIT              95733
FTP-Patator             9531
DoS_GoldenEye           8364
DoS_Slowhttptest        6860
SSH-Patator             5949
Botnet_ARES             5508
DoS_Slowloris           5177
Web_Brute_Force         2734
Web_XSS                 1358


In [17]:
def preprocess_features(data):
    """Select top-20 features, clean NaN/inf, ensure numeric."""
    print("[*] Preprocessing features...")
    available = [f for f in TOP_20_FEATURES if f in data.columns]
    missing = [f for f in TOP_20_FEATURES if f not in data.columns]
    if missing:
        print(f"  WARNING: Missing features: {missing}")

    if len(available) < 10:
        print("  WARNING: Too few named features - falling back to all numeric cols.")
        numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
        exclude = ["flow_id", "src_ip", "dst_ip", "timestamp", "Label"]
        available = [c for c in numeric_cols if c not in exclude][:20]

    print(f"  Using {len(available)} features: {available}")
    X = data[available].copy()
    y = data["Label"].copy()

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

    print(f"  Feature matrix shape: {X.shape}")
    return X, y, available


X, y, feature_names = preprocess_features(data)

[*] Preprocessing features...
  Using 20 features: ['duration', 'fwd_packets_count', 'bwd_packets_count', 'fwd_total_payload_bytes', 'bwd_total_payload_bytes', 'fwd_payload_bytes_max', 'fwd_payload_bytes_mean', 'bwd_payload_bytes_max', 'bwd_payload_bytes_mean', 'bytes_rate', 'packets_rate', 'packets_IAT_mean', 'packet_IAT_max', 'fwd_packets_IAT_total', 'fwd_packets_IAT_mean', 'bwd_packets_IAT_total', 'bwd_packets_IAT_mean', 'fwd_psh_flag_counts', 'fwd_packets_rate', 'bwd_packets_rate']
  Feature matrix shape: (15492, 20)


In [18]:
SAMPLE_SIZE = 200_000   # Set to None to use ALL data (slower but more accurate)
TEST_SIZE = 0.2
RANDOM_STATE = 42

print("[*] Preparing training data...")

if SAMPLE_SIZE and len(X) > SAMPLE_SIZE:
    print(f"  Sampling {SAMPLE_SIZE:,} from {len(X):,} records...")
    idx = np.random.RandomState(RANDOM_STATE).choice(
        len(X), SAMPLE_SIZE, replace=False
    )
    X_sampled = X.iloc[idx]
    y_sampled = y.iloc[idx]
else:
    X_sampled = X
    y_sampled = y

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_sampled)
print(f"  Classes: {list(label_encoder.classes_)}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sampled)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

print(f"  Train: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")

[*] Preparing training data...
  Classes: ['Brute Force', 'DDoS']
  Train: 12,393   Test: 3,099


In [19]:
N_ESTIMATORS = 100

print(f"[*] Training Random Forest (n_estimators={N_ESTIMATORS})...")
print(f"    Training samples : {X_train.shape[0]:,}")
print(f"    Features         : {X_train.shape[1]}")

model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    class_weight="balanced",
)

model.fit(X_train, y_train)
print("\nTraining complete!")

[*] Training Random Forest (n_estimators=100)...
    Training samples : 12,393
    Features         : 20

Training complete!


In [20]:
print("[*] Evaluating model...\n")
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  F1 (wtd)  : {f1:.4f}")

print("\n" + "-"*60)
print("  Classification Report")
print("-"*60)
report_str = classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_,
    digits=4,
)
print(report_str)

print("-"*60)
print("  Confusion Matrix")
print("-"*60)
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Store metrics for metadata file
metrics = {
    "accuracy": float(accuracy),
    "f1_score": float(f1),
    "classification_report": classification_report(
        y_test, y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
    ),
}

[*] Evaluating model...

  Accuracy  : 1.0000  (100.00%)
  F1 (wtd)  : 1.0000

------------------------------------------------------------
  Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

 Brute Force     1.0000    1.0000    1.0000      3097
        DDoS     1.0000    1.0000    1.0000         2

    accuracy                         1.0000      3099
   macro avg     1.0000    1.0000    1.0000      3099
weighted avg     1.0000    1.0000    1.0000      3099

------------------------------------------------------------
  Confusion Matrix
------------------------------------------------------------
[[3097    0]
 [   0    2]]


In [21]:
print(f"[*] Saving model artifacts to '{MODELS_DIR}/'...\n")

# Save model
joblib.dump(model, os.path.join(MODELS_DIR, "rf_model.pkl"))
print("    > rf_model.pkl")

# Save scaler
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.pkl"))
print("    > scaler.pkl")

# Save label encoder
joblib.dump(label_encoder, os.path.join(MODELS_DIR, "label_encoder.pkl"))
print("    > label_encoder.pkl")

# Save metadata
metadata = {
    "feature_names": feature_names,
    "n_features": len(feature_names),
    "classes": list(label_encoder.classes_),
    "n_classes": len(label_encoder.classes_),
    "model_type": "RandomForestClassifier",
    "n_estimators": model.n_estimators,
    "metrics": metrics,
    "attack_mapping": ATTACK_MAPPING,
    "severity_mapping": SEVERITY_MAPPING,
}
with open(os.path.join(MODELS_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("    > model_metadata.json")

print("\n" + "="*60)
print("  Training Pipeline Complete!")
print(f"     Accuracy : {metrics['accuracy']*100:.2f}%")
print(f"     F1 Score : {metrics['f1_score']:.4f}")
print("="*60)

[*] Saving model artifacts to 'models/'...

    > rf_model.pkl
    > scaler.pkl
    > label_encoder.pkl
    > model_metadata.json

  Training Pipeline Complete!
     Accuracy : 100.00%
     F1 Score : 1.0000


In [22]:
print(f"[*] Saving model artifacts to '{MODELS_DIR}/'...\n")

# Save model
joblib.dump(model, os.path.join(MODELS_DIR, "rf_model.pkl"))
print("    > rf_model.pkl")

# Save scaler
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.pkl"))
print("    > scaler.pkl")

# Save label encoder
joblib.dump(label_encoder, os.path.join(MODELS_DIR, "label_encoder.pkl"))
print("    > label_encoder.pkl")

# Save metadata
metadata = {
    "feature_names": feature_names,
    "n_features": len(feature_names),
    "classes": list(label_encoder.classes_),
    "n_classes": len(label_encoder.classes_),
    "model_type": "RandomForestClassifier",
    "n_estimators": model.n_estimators,
    "metrics": metrics,
    "attack_mapping": ATTACK_MAPPING,
    "severity_mapping": SEVERITY_MAPPING,
}
with open(os.path.join(MODELS_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("    > model_metadata.json")

print("\n" + "="*60)
print("  Training Pipeline Complete!")
print(f"     Accuracy : {metrics['accuracy']*100:.2f}%")
print(f"     F1 Score : {metrics['f1_score']:.4f}")
print("="*60)

[*] Saving model artifacts to 'models/'...

    > rf_model.pkl
    > scaler.pkl
    > label_encoder.pkl
    > model_metadata.json

  Training Pipeline Complete!
     Accuracy : 100.00%
     F1 Score : 1.0000
